# `klygo.config` (`mapping.py`): Configuration merge and update operations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IchigoMazone/klygo/blob/v2-release/tutorials/config/Document_Config_Mapping.ipynb)


Combine multiple configuration layers from left to right or apply deep/shallow updates without mutating any input mappings.

In [ ]:
!pip install --upgrade --no-cache-dir git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare workspace

In [1]:
from tempfile import TemporaryDirectory
from klygo import config, files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-config-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-config-102k62bo


## Jump to an API

[`merge`](#merge) · [`update`](#update)

<a id="merge"></a>

### `config.merge`

Merge configurations from left to right without mutating inputs.

**Signature:** `config.merge(*configs: Mapping[str, Any] | box.Box, deep: bool = True) -> box.Box`

In [2]:
defaults_layer = {"model": {"name": "yolov8n", "batch": 16}, "seed": 1}
env_layer = {"model": {"batch": 32}}
cli_layer = {"model": {"lr": 0.001}, "seed": 42}
combined = config.merge(defaults_layer, env_layer, cli_layer)
print("Merged:", combined.to_dict())
print("Original defaults_layer unchanged:", defaults_layer["model"]["batch"])

Merged: {'model': {'name': 'yolov8n', 'batch': 32, 'lr': 0.001}, 'seed': 42}
Original defaults_layer unchanged: 16


In [3]:
# Error example: passing a non-mapping layer
try:
    config.merge({"a": 1}, ["not-a-mapping"])
except TypeError as exc:
    print("TypeError ->", exc)

TypeError -> configs[1] must be Mapping, Box, got list


<a id="update"></a>

### `config.update`

Return a configuration updated from another mapping (deep or shallow).

**Signature:** `config.update(config_data: Mapping[str, Any] | box.Box, updates: Mapping[str, Any] | box.Box, deep: bool = True) -> box.Box`

In [4]:
base = {"model": {"batch": 16, "epochs": 100}}
deep_res = config.update(base, {"model": {"batch": 64}}, deep=True)
shallow_res = config.update(base, {"model": {"batch": 64}}, deep=False)
print("Deep update preserves epochs:", deep_res.model.to_dict())
print("Shallow update replaces model dict:", shallow_res.model.to_dict())

Deep update preserves epochs: {'batch': 64, 'epochs': 100}
Shallow update replaces model dict: {'batch': 64}


In [5]:
# Error example: invalid deep flag type
try:
    config.update(base, {"model": {"batch": 32}}, deep="yes")
except TypeError as exc:
    print("TypeError ->", exc)

TypeError -> deep must be bool, got str


## Put it together

In [6]:
base_cfg = {"model": {"backbone": "resnet50", "lr": 0.01}, "augment": {"flip": True}}
experiment = config.merge(base_cfg, {"model": {"lr": 0.001}})
final_cfg = config.update(experiment, {"augment": {"mixup": 0.2}})
print("Merged + updated:", dict(final_cfg))

Merged + updated: {'model': Box({'backbone': 'resnet50', 'lr': 0.001}), 'augment': Box({'flip': True, 'mixup': 0.2})}


## Remember

- Both `merge()` and `update()` recursively combine nested mappings into a new mapping without mutating the original inputs.
- Next in `klygo.config`: continue with [Document_Config_Access.ipynb](Document_Config_Access.ipynb) or return to [Document_Config_Overview.ipynb](Document_Config_Overview.ipynb).
- Related Klygo modules: [`klygo.files`](../files/Document_Files_Overview.ipynb) · [`klygo.archive`](../archive/Document_Archive_Overview.ipynb).